# 03 - Deep Plant Allergy Benchmark

This notebook retrains the published Deep Plant Allergy architecture on the XAllergen training split, evaluates it on the held-out XAllergen test split, and saves standardized benchmark artifacts for downstream comparison. This benchmark now uses the original Deep Plant ESM-1b embedding family for a paper-faithful comparison.

If the notebook is running outside a checked-out repo, it bootstraps a runtime workspace under `/content/XAllergen2.0` and downloads the required CSVs automatically.

Use this notebook as the external benchmark baseline before running centralized probing in `06_generate_probe_rows.ipynb`.


In [ ]:
import gc
import json
import os
import random
import time
import urllib.request
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, RandomSampler
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"
DATA_FILES = [
    "deepalgpro_train_cleaned.csv",
    "deepalgpro_test_cleaned.csv",
    "positives_splitA.csv",
    "positives_splitB.csv",
    "negatives_splitA.csv",
    "negatives_splitB.csv",
]

try:
    import google.colab  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _looks_like_project_root(candidate: Path) -> bool:
    return (candidate / "data").exists() and (candidate / "notebooks").exists()


def find_local_project_root(start: Path) -> Path | None:
    candidates = []
    env_root = os.environ.get("XALLERGEN_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([start, *start.parents, Path.cwd(), *Path.cwd().parents, Path.home() / "XAllergen2.0"])
    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except FileNotFoundError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_project_root(candidate):
            return candidate
    return None


def ensure_project_dirs(project_root: Path) -> None:
    for relative_path in [
        "data",
        "outputs/deep_plant_allergy_benchmark",
        "models",
        "results/classification",
    ]:
        (project_root / relative_path).mkdir(parents=True, exist_ok=True)


def ensure_required_data_files(project_root: Path) -> None:
    data_dir = project_root / "data"
    for fname in DATA_FILES:
        target = data_dir / fname
        if not target.exists():
            print(f"Downloading missing data file to {target}")
            urllib.request.urlretrieve(f"{RAW}/data/{fname}", target)


def mount_google_drive_project_root() -> Path:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    project_root = Path("/content/drive/MyDrive/XAllergen2.0")
    ensure_project_dirs(project_root)
    ensure_required_data_files(project_root)
    return project_root


def bootstrap_runtime_project_root() -> Path:
    runtime_root = Path("/content/XAllergen2.0")
    ensure_project_dirs(runtime_root)
    ensure_required_data_files(runtime_root)
    return runtime_root


if IN_COLAB:
    try:
        PROJECT_ROOT = mount_google_drive_project_root()
        print(f"Google Colab detected. Using Google Drive project root: {PROJECT_ROOT}")
    except Exception as exc:
        PROJECT_ROOT = bootstrap_runtime_project_root()
        print(f"Google Colab detected, but Google Drive was unavailable ({exc}).")
        print(f"Falling back to runtime project root: {PROJECT_ROOT}")
else:
    PROJECT_ROOT = find_local_project_root(Path.cwd())
    if PROJECT_ROOT is None:
        PROJECT_ROOT = bootstrap_runtime_project_root()
        print(f"Bootstrapped runtime project root: {PROJECT_ROOT}")
    else:
        print(f"Project root: {PROJECT_ROOT}")


Mounted at /content/drive
Google Colab detected. Using Google Drive project root: /content/drive/MyDrive/XAllergen2.0


## Configuration

In [ ]:
# CSV inputs.
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_CSV = DATA_DIR / "deepalgpro_train_cleaned.csv"
TEST_CSV = DATA_DIR / "deepalgpro_test_cleaned.csv"

# Cached embedding files. Each split gets embeddings, labels, and sequence_ids .npy files.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deep_plant_allergy_benchmark"
TRAIN_PREFIX = f"{OUTPUT_DIR}/deepalgpro_train_90_esm1b"
VAL_PREFIX = f"{OUTPUT_DIR}/deepalgpro_val_10_esm1b"
TEST_PREFIX = f"{OUTPUT_DIR}/deepalgpro_test_esm1b"
REGENERATE_EMBEDDINGS = False

# Standardized benchmark outputs.
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "deep_plant_allergy_benchmark.pt"
CLASSIFICATION_RESULTS_DIR = PROJECT_ROOT / "results" / "classification"
TRAINING_HISTORY_CSV = f"{CLASSIFICATION_RESULTS_DIR}/deep_plant_allergy_benchmark_training_history.csv"
TEST_METRICS_JSON = f"{CLASSIFICATION_RESULTS_DIR}/deep_plant_allergy_benchmark_metrics.json"
TEST_PREDICTIONS_CSV = f"{CLASSIFICATION_RESULTS_DIR}/deep_plant_allergy_benchmark_predictions.csv"

ESM_MODEL_NAME = "facebook/esm1b_t33_650M_UR50S"
BATCH_SIZE = 16
EMBED_BATCH_SIZE = 1  # Advanced users can try 2 on high-VRAM GPUs after confirming the default is stable.
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.0
NUM_EPOCHS = 100
EMBEDDING_DIM = 1280  # facebook/esm1b_t33_650M_UR50S hidden size.
EMBEDDING_DTYPE = np.float16
SEED = 42
PATIENCE = 5
MAX_LENGTH = 1000
TRAIN_FRACTION = 0.90

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
CLASSIFICATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU name: {gpu_name}")
    print(f"Available GPU memory: {total_memory_gb:.2f} GB total")
    torch.cuda.empty_cache()
else:
    print("GPU name: None")
    print("Available GPU memory: None")
print(f"Selected device: {device}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"CHECKPOINT_PATH: {CHECKPOINT_PATH}")
print(f"CLASSIFICATION_RESULTS_DIR: {CLASSIFICATION_RESULTS_DIR}")


torch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB
Available GPU memory: 39.49 GB total
Selected device: cuda
PROJECT_ROOT: /content/drive/MyDrive/XAllergen2.0
OUTPUT_DIR: /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark
CHECKPOINT_PATH: /content/drive/MyDrive/XAllergen2.0/models/deep_plant_allergy_benchmark.pt
CLASSIFICATION_RESULTS_DIR: /content/drive/MyDrive/XAllergen2.0/results/classification


## ESM-1b Embedding Helper

In [ ]:
def clean_protein_sequence(sequence):
    return "".join(str(sequence).strip().split()).upper()


def split_train_val_csv(train_csv, train_fraction=TRAIN_FRACTION, seed=SEED):
    df = pd.read_csv(train_csv)
    required_columns = {"sequence_id", "sequence", "label"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    df = df.copy()
    df["sequence"] = df["sequence"].map(clean_protein_sequence)
    df["label"] = df["label"].astype(int)

    val_fraction = 1.0 - train_fraction
    train_df, val_df = train_test_split(
        df,
        test_size=val_fraction,
        random_state=seed,
        stratify=df["label"],
    )

    train_df = train_df.reset_index(drop=True).copy()
    val_df = val_df.reset_index(drop=True).copy()
    return train_df, val_df


def read_test_csv(test_csv):
    df = pd.read_csv(test_csv)
    required_columns = {"sequence_id", "sequence", "label"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    df = df.copy()
    df["sequence"] = df["sequence"].map(clean_protein_sequence)
    df["label"] = df["label"].astype(int)
    return df.reset_index(drop=True)


def embedding_file_paths(output_prefix):
    return {
        "embeddings": f"{output_prefix}_embeddings.npy",
        "labels": f"{output_prefix}_labels.npy",
        "sequence_ids": f"{output_prefix}_sequence_ids.npy",
    }


def embedding_files_exist(output_prefix, expected_len=None):
    paths = embedding_file_paths(output_prefix)
    if not all(os.path.exists(path) for path in paths.values()):
        return False
    if expected_len is None:
        return True
    embeddings = np.load(paths["embeddings"], mmap_mode="r")
    labels = np.load(paths["labels"], mmap_mode="r")
    sequence_ids = np.load(paths["sequence_ids"], allow_pickle=True)
    return (
        embeddings.shape[0] == expected_len
        and len(labels) == expected_len
        and len(sequence_ids) == expected_len
    )


def describe_embedding_cache(output_prefix, expected_len):
    paths = embedding_file_paths(output_prefix)
    files_present = all(os.path.exists(path) for path in paths.values())
    cache_ready = embedding_files_exist(output_prefix, expected_len=expected_len)
    status = "ready" if cache_ready else "stale/missing"
    print(
        f"Cache check for {output_prefix}: exists={files_present}, "
        f"matches_expected_len={cache_ready}, expected_len={expected_len}"
    )
    return {"paths": paths, "files_present": files_present, "cache_ready": cache_ready, "status": status}


def temporary_embedding_file_paths(final_paths):
    temp_paths = {}
    for key, path in final_paths.items():
        path_obj = Path(path)
        temp_paths[key] = str(path_obj.with_name(f"{path_obj.stem}.tmp{path_obj.suffix}"))
    return temp_paths


def atomic_replace_embedding_files(temp_paths, final_paths):
    for key in final_paths:
        os.replace(temp_paths[key], final_paths[key])


def write_esm1b_embeddings_npy(
    df,
    output_prefix,
    model_name=ESM_MODEL_NAME,
    max_length=MAX_LENGTH,
    batch_size=EMBED_BATCH_SIZE,
    dtype=EMBEDDING_DTYPE,
    regenerate=REGENERATE_EMBEDDINGS,
    device=None,
):
    """Generate ESM-1b residue embeddings and write them incrementally to .npy files."""
    from transformers import AutoModel, AutoTokenizer

    paths = embedding_file_paths(output_prefix)
    expected_len = len(df)
    cache_status = describe_embedding_cache(output_prefix, expected_len)
    if cache_status["cache_ready"] and not regenerate:
        print(f"Using existing cached embeddings for {output_prefix}")
        return paths

    if regenerate:
        print(f"REGENERATE_EMBEDDINGS=True, regenerating embeddings for {output_prefix}")
    else:
        print(f"Regenerating stale/missing embeddings for {output_prefix}")

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    sequences = df["sequence"].tolist()
    labels = df["label"].to_numpy(dtype=np.float32)
    sequence_ids = df["sequence_id"].astype(str).to_numpy(dtype=str)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    hidden_size = model.config.hidden_size
    if hidden_size != EMBEDDING_DIM:
        raise ValueError(f"Model hidden size {hidden_size} does not match EMBEDDING_DIM={EMBEDDING_DIM}")

    temp_paths = temporary_embedding_file_paths(paths)
    embeddings_mm = np.lib.format.open_memmap(
        temp_paths["embeddings"],
        mode="w+",
        dtype=dtype,
        shape=(len(sequences), max_length, hidden_size),
    )
    labels_mm = np.lib.format.open_memmap(
        temp_paths["labels"],
        mode="w+",
        dtype=np.float32,
        shape=(len(sequences),),
    )

    try:
        for start in tqdm(range(0, len(sequences), batch_size), desc=f"Writing {os.path.basename(output_prefix)}"):
            end = min(start + batch_size, len(sequences))
            batch_sequences = sequences[start:end]

            encoded = tokenizer(
                batch_sequences,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length + 2,
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}

            with torch.no_grad():
                outputs = model(**encoded)

            hidden_states = outputs.last_hidden_state.detach().cpu()
            attention_masks = encoded["attention_mask"].detach().cpu()

            for offset, (sequence, hidden_state, attention_mask) in enumerate(
                zip(batch_sequences, hidden_states, attention_masks)
            ):
                row = start + offset
                valid_length = int(attention_mask.sum().item())
                residue_embeddings = hidden_state[1:valid_length - 1]
                residue_embeddings = residue_embeddings[:min(len(sequence), max_length)]
                seq_len = min(residue_embeddings.shape[0], max_length)

                embeddings_mm[row, :, :] = 0
                embeddings_mm[row, :seq_len, :] = residue_embeddings.numpy().astype(dtype)
                labels_mm[row] = labels[row]

            embeddings_mm.flush()
            labels_mm.flush()

        np.save(temp_paths["sequence_ids"], sequence_ids)
        embeddings_mm.flush()
        labels_mm.flush()
        del embeddings_mm, labels_mm
        atomic_replace_embedding_files(temp_paths, paths)
    finally:
        for temp_path in temp_paths.values():
            if os.path.exists(temp_path):
                try:
                    os.remove(temp_path)
                except OSError:
                    pass

    print(f"Saved embeddings to {paths['embeddings']}")
    print(f"Saved labels to {paths['labels']}")
    print(f"Saved sequence IDs to {paths['sequence_ids']}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return paths


## Reproducibility

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

## Full Model

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert self.head_dim * heads == embed_size, "Embedding size must be divisible by heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)
        self.attention_weights = None

    def forward(self, values, keys, query, mask=None):
        batch_size = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        values = values.reshape(batch_size, value_len, self.heads, self.head_dim)
        keys = keys.reshape(batch_size, key_len, self.heads, self.head_dim)
        queries = query.reshape(batch_size, query_len, self.heads, self.head_dim)

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = F.softmax(energy / (self.embed_size ** 0.5), dim=3)
        self.attention_weights = attention
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            batch_size, query_len, self.embed_size
        )

        return self.fc_out(out)

    def get_attention_weights(self):
        return self.attention_weights


class EnhancedProteinModelFull(nn.Module):
    def __init__(
        self,
        embedding_dim,
        hidden_dim,
        output_dim,
        num_lstm_layers=2,
        num_fc_layers=2,
        num_attention_heads=16,
        num_filters=64,
        kernel_size=3,
    ):
        super().__init__()

        self.conv1d = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=num_filters,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
        )
        self.batch_norm_conv = nn.BatchNorm1d(num_filters)

        self.lstm = nn.LSTM(
            input_size=num_filters,
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=0.5,
            bidirectional=True,
        )

        self.attention = SelfAttention(embed_size=hidden_dim * 2, heads=num_attention_heads)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc_layers = nn.ModuleList()
        self.batch_norm_fc = nn.ModuleList()
        input_dim = hidden_dim * 2

        for _ in range(num_fc_layers):
            self.fc_layers.append(nn.Linear(input_dim, hidden_dim))
            self.batch_norm_fc.append(nn.BatchNorm1d(hidden_dim))
            input_dim = hidden_dim

        self.fc_output = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv1d(x)
        x = self.batch_norm_conv(x)
        x = x.permute(0, 2, 1)

        x, _ = self.lstm(x)
        x = self.attention(x, x, x)
        x = self.pool(x.permute(0, 2, 1)).squeeze(-1)

        for fc, bn in zip(self.fc_layers, self.batch_norm_fc):
            x = self.relu(bn(fc(x)))
            x = self.dropout(x)

        return self.fc_output(x)

## Data Loading

In [ ]:
def prepare_embedding_files():
    train_df, val_df = split_train_val_csv(TRAIN_CSV, TRAIN_FRACTION, SEED)
    test_df = read_test_csv(TEST_CSV)

    print(f"Train split: {len(train_df)} sequences")
    print(f"Validation split: {len(val_df)} sequences")
    print(f"Held-out test set: {len(test_df)} sequences")

    split_configs = [
        ("train", train_df, TRAIN_PREFIX),
        ("validation", val_df, VAL_PREFIX),
        ("test", test_df, TEST_PREFIX),
    ]
    for split_name, split_df, split_prefix in split_configs:
        cache_status = describe_embedding_cache(split_prefix, expected_len=len(split_df))
        action = "using existing cached embeddings" if cache_status["cache_ready"] and not REGENERATE_EMBEDDINGS else "regenerating stale/missing embeddings"
        print(
            f"{split_name.capitalize()} embeddings: "
            f"exists={cache_status['files_present']}, "
            f"matches_expected_len={cache_status['cache_ready']}, "
            f"expected_len={len(split_df)}, action={action}"
        )

    train_paths = write_esm1b_embeddings_npy(train_df, TRAIN_PREFIX)
    val_paths = write_esm1b_embeddings_npy(val_df, VAL_PREFIX)
    test_paths = write_esm1b_embeddings_npy(test_df, TEST_PREFIX)

    return train_paths, val_paths, test_paths


class NpyEmbeddingDataset(Dataset):
    def __init__(self, paths):
        self.embeddings = np.load(paths["embeddings"], mmap_mode="r")
        self.labels = np.load(paths["labels"], mmap_mode="r")
        self.sequence_ids = np.load(paths["sequence_ids"], allow_pickle=True).astype(str)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        embedding = torch.from_numpy(np.asarray(self.embeddings[idx], dtype=np.float32))
        label = torch.tensor([self.labels[idx]], dtype=torch.float32)
        return embedding, label


def build_dataloader(paths, batch_size, shuffle=False, seed=SEED):
    dataset = NpyEmbeddingDataset(paths)
    if shuffle:
        sampler = RandomSampler(dataset, generator=torch.Generator().manual_seed(seed))
        loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    else:
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return loader, dataset.sequence_ids


def build_dataloaders(train_paths, val_paths, test_paths, batch_size, seed):
    train_loader, train_ids = build_dataloader(train_paths, batch_size, shuffle=True, seed=seed)
    val_loader, val_ids = build_dataloader(val_paths, batch_size, shuffle=False, seed=seed)
    test_loader, test_ids = build_dataloader(test_paths, batch_size, shuffle=False, seed=seed)
    return train_loader, val_loader, test_loader, train_ids, val_ids, test_ids


## Training

In [ ]:
def empty_metrics():
    return {
        "train_losses": [],
        "val_losses": [],
        "train_accuracies": [],
        "val_accuracies": [],
        "train_f1s": [],
        "val_f1s": [],
        "train_mccs": [],
        "val_mccs": [],
        "train_precisions": [],
        "val_precisions": [],
        "train_recalls": [],
        "val_recalls": [],
        "train_roc_aucs": [],
        "val_roc_aucs": [],
        "learning_rates": [],
        "cpu_times": [],
        "cpu_mems": [],
        "gpu_mem_trains": [],
        "gpu_mem_vals": [],
        "infer_times": [],
        "epoch_times": [],
        "tps": [],
        "tns": [],
        "fps": [],
        "fns": [],
    }


def binary_roc_auc(labels, probs):
    try:
        return roc_auc_score(labels, probs)
    except ValueError:
        return np.nan


def evaluate_model(model, data_loader, criterion=None, split_name="Test"):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []
    infer_start_time = time.time()

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)

            if criterion is not None:
                running_loss += criterion(outputs, labels).item()

            probs = torch.sigmoid(outputs).cpu().numpy().ravel()
            preds = (probs > 0.5).astype(int)
            all_labels.extend(labels.detach().cpu().numpy().ravel())
            all_preds.extend(preds)
            all_probs.extend(probs)

    loss = running_loss / len(data_loader) if criterion is not None else np.nan
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()
    results = {
        "Split": split_name,
        "Loss": loss,
        "Accuracy": accuracy_score(all_labels, all_preds),
        "F1 Score": f1_score(all_labels, all_preds, average="weighted"),
        "MCC": matthews_corrcoef(all_labels, all_preds),
        "Precision": precision_score(all_labels, all_preds, average="weighted", zero_division=0),
        "Recall": recall_score(all_labels, all_preds, average="weighted"),
        "ROC-AUC": binary_roc_auc(all_labels, all_probs),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Inference Time (s)": time.time() - infer_start_time,
    }
    return results, np.asarray(all_probs), np.asarray(all_preds), np.asarray(all_labels)


def train_model_with_metrics(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    num_epochs,
    seed,
    patience=PATIENCE,
):
    metrics = empty_metrics()
    best_val_loss = float("inf")
    best_epoch = 0
    best_model_weights = None
    best_model_path = None
    patience_counter = 0
    process = psutil.Process(os.getpid())

    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        start_cpu = time.process_time()

        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        running_train_loss = 0.0
        all_train_labels = []
        all_train_preds = []
        all_train_probs = []

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()
            probs = torch.sigmoid(outputs).detach().cpu().numpy().ravel()
            preds = (probs > 0.5).astype(int)
            all_train_labels.extend(labels.detach().cpu().numpy().ravel())
            all_train_preds.extend(preds)
            all_train_probs.extend(probs)

        gpu_mem_train_mb = 0.0
        if device.type == "cuda" and torch.cuda.is_available():
            gpu_mem_train_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

        train_loss = running_train_loss / len(train_loader)
        train_accuracy = accuracy_score(all_train_labels, all_train_preds)
        train_f1 = f1_score(all_train_labels, all_train_preds, average="weighted")
        train_mcc = matthews_corrcoef(all_train_labels, all_train_preds)
        train_precision = precision_score(all_train_labels, all_train_preds, average="weighted", zero_division=0)
        train_recall = recall_score(all_train_labels, all_train_preds, average="weighted")
        train_roc_auc = binary_roc_auc(all_train_labels, all_train_probs)

        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        val_results, _, _, _ = evaluate_model(model, val_loader, criterion, split_name="Validation")
        gpu_mem_val_mb = 0.0
        if device.type == "cuda" and torch.cuda.is_available():
            gpu_mem_val_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

        val_loss = val_results["Loss"]
        cpu_mem_mb = process.memory_info().rss / (1024 ** 2)
        epoch_time = time.time() - epoch_start_time
        cpu_time = time.process_time() - start_cpu

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
            f"Train Acc: {train_accuracy:.4f}, Val Acc: {val_results['Accuracy']:.4f}, "
            f"Train ROC-AUC: {train_roc_auc:.4f}, Val ROC-AUC: {val_results['ROC-AUC']:.4f}, "
            f"TP: {val_results['TP']}, TN: {val_results['TN']}, "
            f"FP: {val_results['FP']}, FN: {val_results['FN']}, "
            f"CPU Mem: {cpu_mem_mb:.1f} MB, "
            f"GPU Train Mem: {gpu_mem_train_mb:.1f} MB, "
            f"GPU Val Mem: {gpu_mem_val_mb:.1f} MB, "
            f"Infer Time: {val_results['Inference Time (s)']:.2f}s, Epoch Time: {epoch_time:.2f}s"
        )

        metrics["train_losses"].append(train_loss)
        metrics["train_accuracies"].append(train_accuracy)
        metrics["train_f1s"].append(train_f1)
        metrics["train_mccs"].append(train_mcc)
        metrics["train_precisions"].append(train_precision)
        metrics["train_recalls"].append(train_recall)
        metrics["train_roc_aucs"].append(train_roc_auc)
        metrics["val_losses"].append(val_loss)
        metrics["val_accuracies"].append(val_results["Accuracy"])
        metrics["val_f1s"].append(val_results["F1 Score"])
        metrics["val_mccs"].append(val_results["MCC"])
        metrics["val_precisions"].append(val_results["Precision"])
        metrics["val_recalls"].append(val_results["Recall"])
        metrics["val_roc_aucs"].append(val_results["ROC-AUC"])
        metrics["cpu_mems"].append(cpu_mem_mb)
        metrics["gpu_mem_trains"].append(gpu_mem_train_mb)
        metrics["gpu_mem_vals"].append(gpu_mem_val_mb)
        metrics["infer_times"].append(val_results["Inference Time (s)"])
        metrics["epoch_times"].append(epoch_time)
        metrics["cpu_times"].append(cpu_time)
        metrics["tps"].append(val_results["TP"])
        metrics["tns"].append(val_results["TN"])
        metrics["fps"].append(val_results["FP"])
        metrics["fns"].append(val_results["FN"])

        if scheduler:
            scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        metrics["learning_rates"].append(current_lr)
        print(f"Current Learning Rate: {current_lr}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model_weights = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_model_path = CHECKPOINT_PATH
            torch.save({
                "state_dict": best_model_weights,
                "metadata": {
                    "architecture_hyperparameters": {
                        "embedding_dim": EMBEDDING_DIM,
                        "hidden_dim": 128,
                        "output_dim": 1,
                        "num_lstm_layers": 3,
                        "num_fc_layers": 3,
                        "num_attention_heads": 8,
                        "num_filters": 64,
                        "kernel_size": 5,
                    },
                    "esm_model_name": ESM_MODEL_NAME.replace("facebook/", ""),
                    "hf_model_name": ESM_MODEL_NAME,
                    "embedding_source": "huggingface_transformers",
                    "benchmark_variant": "deep_plant_allergy_faithful_esm1b",
                },
            }, best_model_path)
            print(f"Best model saved at epoch {epoch + 1}: {best_model_path}")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"Early stopping patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered. Stopping training.")
            break

        model.train()

    return metrics, best_model_weights, best_epoch, best_model_path


## Run Training

In [ ]:
train_paths, val_paths, test_paths = prepare_embedding_files()
train_loader, val_loader, test_loader, train_ids, val_ids, test_ids = build_dataloaders(
    train_paths,
    val_paths,
    test_paths,
    BATCH_SIZE,
    SEED,
)

model = EnhancedProteinModelFull(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    output_dim=1,
    num_lstm_layers=3,
    num_filters=64,
    num_attention_heads=8,
    num_fc_layers=3,
    kernel_size=5,
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
)

metrics, best_model_weights, best_epoch, best_model_path = train_model_with_metrics(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    NUM_EPOCHS,
    SEED,
    PATIENCE,
)

print(f"Best epoch: {best_epoch + 1}")
print(f"Best model checkpoint: {best_model_path}")


Train split: 4995 sequences
Validation split: 556 sequences
Held-out test set: 1377 sequences
Cache check for /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_train_90_esm1b: exists=False, matches_expected_len=False, expected_len=4995
Train embeddings: exists=False, matches_expected_len=False, expected_len=4995, action=regenerating stale/missing embeddings
Cache check for /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_val_10_esm1b: exists=False, matches_expected_len=False, expected_len=556
Validation embeddings: exists=False, matches_expected_len=False, expected_len=556, action=regenerating stale/missing embeddings
Cache check for /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_test_esm1b: exists=False, matches_expected_len=False, expected_len=1377
Test embeddings: exists=False, matches_expected_len=False, expected_len=1377, action=regenerating stale/missing embeddings
Cache chec

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/536 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm1b_t33_650M_UR50S
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing deepalgpro_train_90_esm1b:   0%|          | 0/4995 [00:00<?, ?it/s]

...

Saved embeddings to /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_val_10_esm1b_embeddings.npy
Saved labels to /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_val_10_esm1b_labels.npy
Saved sequence IDs to /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_val_10_esm1b_sequence_ids.npy
Cache check for /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_test_esm1b: exists=False, matches_expected_len=False, expected_len=1377
Regenerating stale/missing embeddings for /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_test_esm1b


Loading weights:   0%|          | 0/536 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm1b_t33_650M_UR50S
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing deepalgpro_test_esm1b:   0%|          | 0/1377 [00:00<?, ?it/s]

Saved embeddings to /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_test_esm1b_embeddings.npy
Saved labels to /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_test_esm1b_labels.npy
Saved sequence IDs to /content/drive/MyDrive/XAllergen2.0/outputs/deep_plant_allergy_benchmark/deepalgpro_test_esm1b_sequence_ids.npy
Epoch [1/100] Train Loss: 0.6878, Val Loss: 0.6284, Train Acc: 0.5548, Val Acc: 0.7572, Train ROC-AUC: 0.5822, Val ROC-AUC: 0.8452, TP: 227, TN: 194, FP: 89, FN: 46, CPU Mem: 15397.3 MB, GPU Train Mem: 2488.1 MB, GPU Val Mem: 2154.7 MB, Infer Time: 4.36s, Epoch Time: 433.34s
Current Learning Rate: 5e-06
Best model saved at epoch 1: /content/drive/MyDrive/XAllergen2.0/models/deep_plant_allergy_benchmark.pt
Epoch [2/100] Train Loss: 0.6373, Val Loss: 0.5848, Train Acc: 0.6270, Val Acc: 0.8363, Train ROC-AUC: 0.6869, Val ROC-AUC: 0.9088, TP: 233, TN: 232, FP: 51, FN: 40, CPU Mem: 15402.4 MB, GPU Train Mem: 248

## Save Metrics

In [ ]:
train_metrics = pd.DataFrame({
    "Epoch": range(1, len(metrics["train_losses"]) + 1),
    "Train Loss": metrics["train_losses"],
    "Train Accuracy": metrics["train_accuracies"],
    "Train F1 Score": metrics["train_f1s"],
    "Train MCC": metrics["train_mccs"],
    "Train Precision": metrics["train_precisions"],
    "Train Recall": metrics["train_recalls"],
    "Train ROC-AUC": metrics["train_roc_aucs"],
    "Learning Rate": metrics["learning_rates"],
})

val_metrics = pd.DataFrame({
    "Epoch": range(1, len(metrics["val_losses"]) + 1),
    "Validation Loss": metrics["val_losses"],
    "Validation Accuracy": metrics["val_accuracies"],
    "Validation F1 Score": metrics["val_f1s"],
    "Validation MCC": metrics["val_mccs"],
    "Validation Precision": metrics["val_precisions"],
    "Validation Recall": metrics["val_recalls"],
    "Validation ROC-AUC": metrics["val_roc_aucs"],
    "TP": metrics["tps"],
    "TN": metrics["tns"],
    "FP": metrics["fps"],
    "FN": metrics["fns"],
    "CPU Memory (MB)": metrics["cpu_mems"],
    "GPU Memory Train (MB)": metrics["gpu_mem_trains"],
    "GPU Memory Val (MB)": metrics["gpu_mem_vals"],
    "Inference Time (s)": metrics["infer_times"],
    "Epoch Time (s)": metrics["epoch_times"],
    "CPU Time": metrics["cpu_times"],
})

combined_metrics = pd.concat([train_metrics, val_metrics.drop(columns=["Epoch"])], axis=1)
combined_metrics.to_csv(TRAINING_HISTORY_CSV, index=False)
print(f"Saved training history to {TRAINING_HISTORY_CSV}")
combined_metrics.tail()


Saved training history to /content/drive/MyDrive/XAllergen2.0/results/classification/deep_plant_allergy_benchmark_training_history.csv


,Epoch,Train Loss,Train Accuracy,Train F1 Score,Train MCC,Train Precision,Train Recall,Train ROC-AUC,Learning Rate,Validation Loss,...,TP,TN,FP,FN,CPU Memory (MB),GPU Memory Train (MB),GPU Memory Val (MB),Inference Time (s),Epoch Time (s),CPU Time
43,44,0.141365,0.960360,0.960363,0.920749,0.960410,0.960360,0.989871,0.000005,0.159955,...,256,271,12,17,15412.238281,2488.074707,2154.680176,4.014147,47.819619,75.985107
44,45,0.131801,0.965966,0.965966,0.931907,0.965966,0.965966,0.991702,0.000005,0.164628,...,257,271,12,16,15412.238281,2488.074707,2154.680176,4.005908,47.871764,76.383150
45,46,0.131423,0.964765,0.964765,0.929504,0.964765,0.964765,0.991658,0.000005,0.160363,...,259,270,13,14,15412.238281,2488.074707,2154.680176,4.000503,47.753967,76.328226
46,47,0.121378,0.966967,0.966969,0.933969,0.967020,0.966967,0.993391,0.000005,0.166149,...,258,269,14,15,15412.238281,2488.074707,2154.680176,3.995315,48.922582,77.919041
47,48,0.125871,0.966767,0.966767,0.933515,0.966771,0.966767,0.991616,0.000005,0.159340,...,258,269,14,15,15412.238281,2488.074707,2154.680176,3.983147,47.526268,75.705187


## Test Best Model


In [ ]:
def to_json_compatible(value):
    if isinstance(value, dict):
        return {key: to_json_compatible(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_compatible(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


best_model = EnhancedProteinModelFull(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    output_dim=1,
    num_lstm_layers=3,
    num_filters=64,
    num_attention_heads=8,
    num_fc_layers=3,
    kernel_size=5,
).to(device)

checkpoint_payload = torch.load(best_model_path, map_location=device, weights_only=False)
state_dict = checkpoint_payload["state_dict"] if isinstance(checkpoint_payload, dict) and "state_dict" in checkpoint_payload else checkpoint_payload
best_model.load_state_dict(state_dict)
test_results, test_probs, test_preds, test_labels = evaluate_model(
    best_model,
    test_loader,
    criterion,
    split_name="Test",
)

test_results_payload = to_json_compatible({
    "model_name": "deep_plant_allergy_benchmark",
    "benchmark_type": "published_architecture_retrained_on_xallergen",
    "embedding_model": ESM_MODEL_NAME,
    "seed": SEED,
    "best_epoch": int(best_epoch + 1),
    "checkpoint_path": str(CHECKPOINT_PATH),
    **test_results,
})
Path(TEST_METRICS_JSON).write_text(json.dumps(test_results_payload, indent=2))
pd.DataFrame({
    "sequence_id": test_ids,
    "label": test_labels.astype(int),
    "probability": test_probs,
    "prediction": test_preds.astype(int),
}).to_csv(TEST_PREDICTIONS_CSV, index=False)

print(f"Saved benchmark metrics to {TEST_METRICS_JSON}")
print(f"Saved test predictions to {TEST_PREDICTIONS_CSV}")
pd.DataFrame([test_results_payload])


Saved benchmark metrics to /content/drive/MyDrive/XAllergen2.0/results/classification/deep_plant_allergy_benchmark_metrics.json
Saved test predictions to /content/drive/MyDrive/XAllergen2.0/results/classification/deep_plant_allergy_benchmark_predictions.csv


,model_name,benchmark_type,embedding_model,seed,best_epoch,checkpoint_path,Split,Loss,Accuracy,F1 Score,MCC,Precision,Recall,ROC-AUC,TP,TN,FP,FN,Inference Time (s)
0,deep_plant_allergy_benchmark,published_architecture_retrained_on_xallergen,facebook/esm1b_t33_650M_UR50S,42,43,/content/drive/MyDrive/XAllergen2.0/models/dee...,Test,0.18638,0.938998,0.939004,0.878019,0.939074,0.938998,0.97424,634,659,46,38,10.631755


In [ ]:
# %% [markdown]
# ## Optional: Disconnect Colab Runtime After Successful Completion

# %%
AUTO_DISCONNECT_COLAB = True

if AUTO_DISCONNECT_COLAB and IN_COLAB:
    print("Notebook finished successfully. Disconnecting and deleting Colab runtime to stop GPU usage.")
    from google.colab import runtime  # type: ignore
    runtime.unassign()

Notebook finished successfully. Disconnecting and deleting Colab runtime to stop GPU usage.
